# 📚 Book Genre Classification with RoBERTa-base
### 12-Class Reduced Dataset

| | |
|---|---|
| **Dataset** | `BooksClassifier_dataset_reduced_12class.csv` — 6,489 books, 12 genres |
| **Text column** | `model_text` (title + cleaned description, avg ~735 chars) |
| **Model** | `roberta-base` (HuggingFace Transformers) |
| **Task** | Multi-class text classification |

## Dataset — 12 Genre Classes

| Genre | Approx. samples |
|---|---|
| Speculative Fiction | 900 |
| Crime and Suspense | 900 |
| History and Biography | 900 |
| Romance | 754 |
| Mind and Self Improvement | 670 |
| Travel and Adventure | 477 |
| Children and Young Adult | 473 |
| Graphic and Short Form | 454 |
| Literary Fiction | 406 |
| Philosophy | 313 |
| Cooking | 163 |
| Science | 79 |

> ⚠️ **Class imbalance is significant** (Science: 79 vs top classes: 900). Class-weighted loss is enabled by default to compensate.

## Why RoBERTa-base?

RoBERTa (**R**obustly **O**ptimized **BERT** Pre-training **A**pproach) was developed by Facebook AI Research as a direct evolution of BERT — trained on 160 GB of text (10× BERT), without the Next Sentence Prediction objective, and with dynamic masking during pre-training.

### Key Advantages for Book Genre Classification

1. **Strong Contextual Understanding** — 12 Transformer layers with 12 attention heads capture long-range semantic relationships critical for mixed-signal genre descriptions.
2. **High Text Classification Accuracy** — Consistently 1–3% above vanilla BERT on sentence/document classification, thanks to its 50k BPE vocabulary.
3. **Rich Pre-trained Representations** — Genre-indicative phrases like `"spaceship"`, `"brooding duke"`, `"serial killer"`, `"philosophical inquiry"` are already encoded in contextually rich embeddings before fine-tuning.
4. **Efficient Fine-tuning** — A single linear head on the `[CLS]` token adds only ~12k parameters for 12 classes. Stable fine-tuning in 3–5 epochs.

### Expected Performance After Tuning

| Metric | Expected Range |
|---|---|
| Macro F1-score | 0.78 – 0.86 |
| Weighted F1-score | 0.84 – 0.90 |
| Overall Accuracy | 84% – 90% |

> Performance on `Science` (79 samples) and `Cooking` (163 samples) will benefit most from class weighting but will remain noisier due to low sample counts.

---
## 1. Imports & Dependencies

In [ ]:
# ── Standard Library ───────────────────────────────────────────────────────────
import os
import re
import time
import warnings
import random
import logging
from pathlib import Path

# ── Third-Party ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# ── Optional: nice progress bars ───────────────────────────────────────────────
try:
    from tqdm.auto import tqdm
    TQDM_AVAILABLE = True
except ImportError:
    TQDM_AVAILABLE = False
    def tqdm(x, **kwargs):
        return x

warnings.filterwarnings("ignore")
print("✅ All imports successful.")

---
## 2. Configuration

All hyper-parameters and paths live in a single `Config` class.

**Key changes from the 22-class version:**
- `DATA_PATH` → new 12-class CSV
- `TEXT_COLUMN` → `model_text` (prepends title to the cleaned description — richest column)
- `PATIENCE` reduced to 2 (fewer classes converge faster)
- `USE_HIGH_CONFIDENCE_ONLY` flag to optionally filter to the 5,932 high-confidence rows

In [ ]:
class Config:
    # ── Paths ──────────────────────────────────────────────────────────────
    DATA_PATH  = "BooksClassifier_dataset_reduced_12class.csv"
    OUTPUT_DIR = "roberta_book_classifier_12class_output"

    # ── Model ──────────────────────────────────────────────────────────────
    MODEL_NAME   = "roberta-base"
    MAX_LEN      = 256          # covers 75th pct of description lengths
    TEXT_COLUMN  = "model_text" # title + cleaned description (avg ~735 chars)
    LABEL_COLUMN = "target_genre"

    # ── Training hyper-parameters ─────────────────────────────────────────
    BATCH_SIZE    = 16           # reduce to 8 if GPU memory is tight
    EPOCHS        = 8
    LEARNING_RATE = 2e-5         # sweet-spot for RoBERTa fine-tuning
    WEIGHT_DECAY  = 0.01
    WARMUP_RATIO  = 0.1          # 10% of total steps for warm-up
    MAX_GRAD_NORM = 1.0          # gradient clipping

    # ── Early stopping ─────────────────────────────────────────────────────
    PATIENCE = 2                 # fewer classes converge faster

    # ── Data splits ────────────────────────────────────────────────────────
    TRAIN_SIZE = 0.80
    VAL_SIZE   = 0.10
    TEST_SIZE  = 0.10

    # ── Reproducibility ────────────────────────────────────────────────────
    SEED = 42

    # ── Class imbalance ────────────────────────────────────────────────────
    # Strongly recommended: Science (79) vs top classes (900) → 11× imbalance
    USE_CLASS_WEIGHTS = True

    # ── Data quality filter ────────────────────────────────────────────────
    # Set True to train only on the 5,932 high-confidence rows
    # Set False to use all 6,489 rows (includes 557 lower-confidence rows)
    USE_HIGH_CONFIDENCE_ONLY = False

    # ── Misc ───────────────────────────────────────────────────────────────
    NUM_WORKERS = 0              # safe default; set to 2-4 on Linux for speed
    PIN_MEMORY  = True


cfg = Config()
print(f"Dataset          : {cfg.DATA_PATH}")
print(f"Text column      : {cfg.TEXT_COLUMN}")
print(f"Label column     : {cfg.LABEL_COLUMN}")
print(f"Output directory : {cfg.OUTPUT_DIR}")
print(f"Model            : {cfg.MODEL_NAME}")
print(f"Max seq length   : {cfg.MAX_LEN}")
print(f"Batch size       : {cfg.BATCH_SIZE}")
print(f"Epochs           : {cfg.EPOCHS}")
print(f"Learning rate    : {cfg.LEARNING_RATE}")
print(f"High-conf only   : {cfg.USE_HIGH_CONFIDENCE_ONLY}")

---
## 3. Utilities

In [ ]:
def set_seed(seed: int) -> None:
    """Fix all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def setup_logging(output_dir: str) -> logging.Logger:
    """Configure a logger that writes to console and a file."""
    os.makedirs(output_dir, exist_ok=True)
    log_path = os.path.join(output_dir, "training.log")

    # Reset handlers to avoid duplicate output on re-runs
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(log_path, mode="w"),
        ],
    )
    return logging.getLogger(__name__)


def get_device() -> torch.device:
    """Return the best available device."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"[Device] GPU detected: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        print("[Device] No GPU found — using CPU (training will be slower).")
    return device


# Initialise
set_seed(cfg.SEED)
device = get_device()
logger = setup_logging(cfg.OUTPUT_DIR)
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

logger.info("=" * 72)
logger.info("  Book Genre Classification — RoBERTa-base  [12-Class Dataset]")
logger.info("=" * 72)

---
## 4. Text Preprocessing

The `model_text` column already contains cleaned, lowercased text. We apply only minimal sanitisation — removing control characters and collapsing whitespace — so RoBERTa's BPE tokeniser can retain full sub-word signal.

In [ ]:
def clean_text(text: str) -> str:
    """
    Minimal, semantics-preserving cleaning.
    • Remove non-printable / control characters
    • Collapse excess whitespace
    We do NOT strip punctuation or apply stemming — RoBERTa's BPE
    tokeniser handles sub-word normalisation.
    """
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Smoke test
sample = "  Hello\x00\x01  World!   "
print(f"Input  : {repr(sample)}")
print(f"Cleaned: {repr(clean_text(sample))}")

---
## 5. PyTorch Dataset

In [ ]:
class BookDataset(Dataset):
    """PyTorch Dataset that tokenises book descriptions on-the-fly."""

    def __init__(
        self,
        texts: list[str],
        labels: list[int],
        tokenizer: RobertaTokenizer,
        max_len: int,
    ) -> None:
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


print("✅ BookDataset defined.")

---
## 6. Data Loading & Preparation

The loader handles the specific column layout of the 12-class dataset:
- Uses `model_text` as the input text (title + description, pre-cleaned)
- Optionally filters to `high_confidence == True` rows only
- Drops duplicates on the text column
- Encodes `target_genre` with `LabelEncoder`

In [ ]:
def load_and_prepare_data(
    path: str, logger: logging.Logger
) -> tuple[pd.DataFrame, LabelEncoder]:
    """
    Load, filter, clean, and encode the 12-class dataset.

    Returns
    -------
    df            : cleaned DataFrame with 'text' and 'label' columns
    label_encoder : fitted LabelEncoder for inverse-transforming predictions
    """
    logger.info(f"Loading dataset from: {path}")
    df = pd.read_csv(path)
    logger.info(f"Raw dataset shape: {df.shape}")

    # ── Optional: keep only high-confidence rows ────────────────────────────
    if cfg.USE_HIGH_CONFIDENCE_ONLY:
        before = len(df)
        df = df[df["high_confidence"] == True].reset_index(drop=True)
        logger.info(
            f"High-confidence filter: {before} → {len(df)} rows "
            f"(removed {before - len(df)} low-confidence)"
        )

    # ── Keep only rows flagged for training (all rows in this dataset) ──────
    df = df[df["use_for_training"] == True].reset_index(drop=True)
    logger.info(f"After use_for_training filter: {df.shape}")

    # ── Missing values ──────────────────────────────────────────────────────
    missing = df[[cfg.TEXT_COLUMN, cfg.LABEL_COLUMN]].isnull().sum()
    if missing.any():
        logger.warning(f"Missing values:\n{missing[missing > 0]}")
        df = df.dropna(subset=[cfg.TEXT_COLUMN, cfg.LABEL_COLUMN])

    # ── Duplicates on text column ───────────────────────────────────────────
    n_dup = df.duplicated(subset=[cfg.TEXT_COLUMN]).sum()
    if n_dup:
        logger.warning(f"Removing {n_dup} duplicate text rows.")
        df = df.drop_duplicates(subset=[cfg.TEXT_COLUMN]).reset_index(drop=True)

    # ── Text cleaning ──────────────────────────────────────────────────────
    df["text"] = df[cfg.TEXT_COLUMN].apply(clean_text)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)

    # ── Label encoding ─────────────────────────────────────────────────────
    le = LabelEncoder()
    df["label"] = le.fit_transform(df[cfg.LABEL_COLUMN])

    # ── Dataset statistics ─────────────────────────────────────────────────
    logger.info(f"\nFinal dataset shape : {df.shape}")
    logger.info(f"Number of classes   : {len(le.classes_)}")
    logger.info(f"Classes             : {list(le.classes_)}")
    logger.info("\nClass distribution:")
    vc = df[cfg.LABEL_COLUMN].value_counts()
    for genre, count in vc.items():
        logger.info(f"  {genre:<30} {count:>4}")

    return df, le


def split_dataset(
    df: pd.DataFrame, logger: logging.Logger
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Stratified 80 / 10 / 10 split.
    Stratification is critical given the extreme imbalance
    (Science: 79 vs top classes: 900).
    """
    train_df, temp_df = train_test_split(
        df,
        test_size=(cfg.VAL_SIZE + cfg.TEST_SIZE),
        stratify=df["label"],
        random_state=cfg.SEED,
    )
    relative_test_size = cfg.TEST_SIZE / (cfg.VAL_SIZE + cfg.TEST_SIZE)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=relative_test_size,
        stratify=temp_df["label"],
        random_state=cfg.SEED,
    )

    logger.info(
        f"\nData split — Train: {len(train_df)} | "
        f"Val: {len(val_df)} | Test: {len(test_df)}"
    )
    return train_df, val_df, test_df


print("✅ Data loading and splitting functions defined.")

In [ ]:
# ── Run data loading ───────────────────────────────────────────────────────────
df, label_encoder = load_and_prepare_data(cfg.DATA_PATH, logger)
num_classes = len(label_encoder.classes_)

print(f"\n{'='*50}")
print(f"  Classes ({num_classes}): {list(label_encoder.classes_)}")
print(f"{'='*50}")

# Preview
df[[cfg.LABEL_COLUMN, "text", "label", "high_confidence"]].head()

---
## 7. Exploratory Data Analysis

In [ ]:
def plot_class_distribution(df: pd.DataFrame, output_dir: str) -> None:
    """Display and save a bar chart of class frequencies."""
    vc = df[cfg.LABEL_COLUMN].value_counts()

    # Colour bars by relative frequency (highlight minority classes in red)
    max_count = vc.max()
    colors = ["#d73027" if c < max_count * 0.25 else
              "#fee090" if c < max_count * 0.5  else
              "steelblue" for c in vc.values]

    fig, ax = plt.subplots(figsize=(14, 5))
    bars = ax.bar(vc.index, vc.values, color=colors, edgecolor="white")
    ax.bar_label(bars, padding=3, fontsize=8)
    ax.set_title("Class Distribution — 12-Class Dataset", fontsize=13)
    ax.set_ylabel("Sample Count")
    ax.set_xlabel("")

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="steelblue", label="≥ 50% of max"),
        Patch(facecolor="#fee090",   label="25–50% of max"),
        Patch(facecolor="#d73027",   label="< 25% of max (minority)"),
    ]
    ax.legend(handles=legend_elements, fontsize=8)

    plt.xticks(rotation=40, ha="right", fontsize=9)
    plt.tight_layout()
    path = os.path.join(output_dir, "class_distribution.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[Saved] Class distribution → {path}")


plot_class_distribution(df, cfg.OUTPUT_DIR)

In [ ]:
# Description length distribution
df["text_len"] = df["text"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Overall histogram
axes[0].hist(df["text_len"], bins=60, color="steelblue", edgecolor="white")
axes[0].axvline(df["text_len"].quantile(0.75), color="red",    linestyle="--", label="75th pct")
axes[0].axvline(df["text_len"].median(),       color="orange", linestyle="--", label="Median")
axes[0].set_title("Text Length Distribution (characters)")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Count")
axes[0].legend()

# Per-genre box plot
genre_order = df.groupby(cfg.LABEL_COLUMN)["text_len"].median().sort_values().index
df.boxplot(
    column="text_len",
    by=cfg.LABEL_COLUMN,
    ax=axes[1],
    order=genre_order,
    vert=False,
    patch_artist=True,
    boxprops=dict(facecolor="steelblue", alpha=0.6),
)
axes[1].set_title("Text Length by Genre")
axes[1].set_xlabel("Characters")
axes[1].set_ylabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()

print("\nText length summary:")
print(df["text_len"].describe().round(0))

In [ ]:
# High-confidence split per genre
conf_df = df.groupby(cfg.LABEL_COLUMN)["high_confidence"].value_counts(normalize=True).unstack(fill_value=0)
conf_df.columns = ["Low Confidence", "High Confidence"] if False in conf_df.columns else ["High Confidence"]

print("High-confidence ratio per genre:")
print(
    df.groupby(cfg.LABEL_COLUMN)["high_confidence"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={True: "High Conf ✓", False: "Low Conf ✗"})
    .to_string()
)

---
## 8. Data Splitting

In [ ]:
train_df, val_df, test_df = split_dataset(df, logger)

print(f"\nSplit summary")
print(f"  Train : {len(train_df):,} samples  ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Val   : {len(val_df):,} samples  ({len(val_df)/len(df)*100:.1f}%)")
print(f"  Test  : {len(test_df):,} samples  ({len(test_df)/len(df)*100:.1f}%)")

# Verify stratification — smallest class (Science ~79 rows)
print(f"\nSmallest class in test set:")
smallest = test_df[cfg.LABEL_COLUMN].value_counts().tail(3)
print(smallest.to_string())

---
## 9. Class Weights

With Science at 79 samples and top classes at 900, the imbalance ratio is ~11×. Class-weighted loss ensures minority genres are not ignored.

In [ ]:
def compute_weights(
    train_labels: np.ndarray, num_classes: int, device: torch.device
) -> torch.Tensor:
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_classes),
        y=train_labels,
    )
    return torch.tensor(weights, dtype=torch.float).to(device)


if cfg.USE_CLASS_WEIGHTS:
    class_weights = compute_weights(train_df["label"].values, num_classes, device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    logger.info("Using class-weighted CrossEntropyLoss to address imbalance.")

    # Show the weight assigned to each class
    print("\nClass weights (higher = rarer class):")
    for i, (cls, w) in enumerate(zip(label_encoder.classes_, class_weights.cpu().numpy())):
        bar = "█" * int(w * 10)
        print(f"  {cls:<30} {w:.3f}  {bar}")
else:
    loss_fn = nn.CrossEntropyLoss()
    logger.info("Using standard CrossEntropyLoss.")

print(f"\nLoss function: {loss_fn}")

---
## 10. Tokeniser & DataLoaders

In [ ]:
logger.info(f"Loading tokeniser: {cfg.MODEL_NAME}")
tokenizer = RobertaTokenizer.from_pretrained(cfg.MODEL_NAME)

train_dataset = BookDataset(
    train_df["text"].tolist(), train_df["label"].tolist(), tokenizer, cfg.MAX_LEN
)
val_dataset = BookDataset(
    val_df["text"].tolist(), val_df["label"].tolist(), tokenizer, cfg.MAX_LEN
)
test_dataset = BookDataset(
    test_df["text"].tolist(), test_df["label"].tolist(), tokenizer, cfg.MAX_LEN
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=cfg.PIN_MEMORY and device.type == "cuda",
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE * 2,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=cfg.PIN_MEMORY and device.type == "cuda",
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.BATCH_SIZE * 2,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=cfg.PIN_MEMORY and device.type == "cuda",
)

# Verify a batch
sample_batch = next(iter(train_loader))
print(f"Batch — input_ids:      {sample_batch['input_ids'].shape}")
print(f"Batch — attention_mask: {sample_batch['attention_mask'].shape}")
print(f"Batch — labels:         {sample_batch['labels'].shape}")
print(f"\nLabel IDs in first batch: {sample_batch['labels'].tolist()}")
print(f"Genres: {[label_encoder.classes_[i] for i in sample_batch['labels'].tolist()]}")

---
## 11. Model Initialisation

In [ ]:
logger.info(f"Loading model: {cfg.MODEL_NAME}  ({num_classes} output classes)")

model = RobertaForSequenceClassification.from_pretrained(
    cfg.MODEL_NAME,
    num_labels=num_classes,
    # Store id↔label mapping in model config for inference convenience
    id2label={i: cls for i, cls in enumerate(label_encoder.classes_)},
    label2id={cls: i for i, cls in enumerate(label_encoder.classes_)},
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)
model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

logger.info(f"Total parameters     : {total_params:,}")
logger.info(f"Trainable parameters : {trainable_params:,}")
print(f"\nModel loaded on: {device}")
print(f"Classification head outputs: {num_classes} classes")

---
## 12. Optimiser & Scheduler

In [ ]:
# Bias and LayerNorm weights are NOT decayed
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_parameters = [
    {
        "params": [
            p for n, p in model.named_parameters()
            if not any(nd in n for nd in no_decay)
        ],
        "weight_decay": cfg.WEIGHT_DECAY,
    },
    {
        "params": [
            p for n, p in model.named_parameters()
            if any(nd in n for nd in no_decay)
        ],
        "weight_decay": 0.0,
    },
]
optimizer = AdamW(optimizer_grouped_parameters, lr=cfg.LEARNING_RATE, eps=1e-8)

total_steps  = len(train_loader) * cfg.EPOCHS
warmup_steps = int(total_steps * cfg.WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

logger.info(f"Optimiser  : AdamW  lr={cfg.LEARNING_RATE}  wd={cfg.WEIGHT_DECAY}")
logger.info(f"Scheduler  : Linear warm-up ({warmup_steps} steps) → linear decay")
logger.info(f"Total steps: {total_steps}  |  Warmup steps: {warmup_steps}")

---
## 13. Training & Evaluation Functions

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: AdamW,
    scheduler,
    loss_fn: nn.CrossEntropyLoss,
    device: torch.device,
    epoch: int,
    logger: logging.Logger,
) -> tuple[float, float]:
    """One full pass over the training set."""
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    loop = tqdm(loader, desc=f"Epoch {epoch} [Train]", leave=False)
    for batch in loop:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = loss_fn(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        total_loss    += loss.item() * labels.size(0)
        preds          = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        if TQDM_AVAILABLE:
            loop.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: nn.CrossEntropyLoss,
    device: torch.device,
    desc: str = "Eval",
) -> tuple[float, float, np.ndarray, np.ndarray]:
    """Evaluate model; returns loss, accuracy, all predictions, all true labels."""
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    all_preds, all_labels = [], []

    loop = tqdm(loader, desc=desc, leave=False)
    for batch in loop:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = loss_fn(logits, labels)

        total_loss    += loss.item() * labels.size(0)
        preds          = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (
        total_loss / total_samples,
        total_correct / total_samples,
        np.array(all_preds),
        np.array(all_labels),
    )


print("✅ train_one_epoch and evaluate functions defined.")

---
## 14. Training Loop with Early Stopping

In [ ]:
best_val_loss   = float("inf")
best_val_acc    = 0.0
epochs_no_imp   = 0
best_model_path = os.path.join(cfg.OUTPUT_DIR, "best_model")

history: dict = {
    "train_loss": [], "train_acc": [],
    "val_loss":   [], "val_acc":   [],
}

logger.info("\n" + "─" * 72)
logger.info("  TRAINING")
logger.info("─" * 72)

for epoch in range(1, cfg.EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scheduler, loss_fn, device, epoch, logger
    )
    val_loss, val_acc, _, _ = evaluate(
        model, val_loader, loss_fn, device, desc=f"Epoch {epoch} [Val]"
    )

    elapsed = time.time() - t0
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    logger.info(
        f"Epoch {epoch:2d}/{cfg.EPOCHS}  "
        f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
        f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  "
        f"[{elapsed:.0f}s]"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc  = val_acc
        epochs_no_imp = 0
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        logger.info(f"  ✓ Best model saved (val_loss={best_val_loss:.4f})")
    else:
        epochs_no_imp += 1
        logger.info(
            f"  ✗ No improvement ({epochs_no_imp}/{cfg.PATIENCE} patience used)"
        )
        if epochs_no_imp >= cfg.PATIENCE:
            logger.info(f"\n[Early Stopping] Triggered at epoch {epoch}.")
            break

logger.info(
    f"\nBest validation — Loss: {best_val_loss:.4f}  Acc: {best_val_acc:.4f}"
)

---
## 15. Training Curves

In [ ]:
def plot_training_curves(history: dict, output_dir: str) -> None:
    """Display and save training/validation loss and accuracy curves."""
    epochs_run = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_run, history["train_loss"], label="Train Loss", marker="o")
    axes[0].plot(epochs_run, history["val_loss"],   label="Val Loss",   marker="s")
    axes[0].set_title("Loss per Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_xticks(list(epochs_run))
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(epochs_run, history["train_acc"], label="Train Acc", marker="o")
    axes[1].plot(epochs_run, history["val_acc"],   label="Val Acc",   marker="s")
    axes[1].set_title("Accuracy per Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_xticks(list(epochs_run))
    axes[1].legend()
    axes[1].grid(True)

    plt.suptitle("RoBERTa Fine-tuning — 12-Class Book Genre", fontsize=12)
    plt.tight_layout()
    path = os.path.join(output_dir, "training_curves.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[Saved] Training curves → {path}")


plot_training_curves(history, cfg.OUTPUT_DIR)

---
## 16. Final Test Evaluation

In [ ]:
logger.info("\n" + "─" * 72)
logger.info("  FINAL TEST EVALUATION  (best checkpoint)")
logger.info("─" * 72)

best_model = RobertaForSequenceClassification.from_pretrained(best_model_path)
best_model.to(device)

test_loss, test_acc, y_pred, y_true = evaluate(
    best_model, test_loader, loss_fn, device, desc="Test"
)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted"
)
macro_f1 = precision_recall_fscore_support(y_true, y_pred, average="macro")[2]

class_names = list(label_encoder.classes_)

print(f"{'─'*50}")
print(f"  Test Loss           : {test_loss:.4f}")
print(f"  Test Accuracy       : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Weighted Precision  : {precision:.4f}")
print(f"  Weighted Recall     : {recall:.4f}")
print(f"  Weighted F1-score   : {f1:.4f}")
print(f"  Macro    F1-score   : {macro_f1:.4f}")
print(f"{'─'*50}")

In [ ]:
# Per-class classification report
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print("Per-class classification report:\n")
print(report)

# Save metrics
metrics_path = os.path.join(cfg.OUTPUT_DIR, "test_metrics.txt")
with open(metrics_path, "w", encoding="utf-8") as f:
    f.write("=" * 72 + "\n")
    f.write("  Book Genre Classifier (12-class) — Test Results\n")
    f.write("=" * 72 + "\n\n")
    f.write(f"Dataset             : {cfg.DATA_PATH}\n")
    f.write(f"Text column         : {cfg.TEXT_COLUMN}\n")
    f.write(f"High-conf only      : {cfg.USE_HIGH_CONFIDENCE_ONLY}\n\n")
    f.write(f"Test Accuracy       : {test_acc:.4f}  ({test_acc*100:.2f}%)\n")
    f.write(f"Weighted Precision  : {precision:.4f}\n")
    f.write(f"Weighted Recall     : {recall:.4f}\n")
    f.write(f"Weighted F1-score   : {f1:.4f}\n")
    f.write(f"Macro    F1-score   : {macro_f1:.4f}\n\n")
    f.write(report)
print(f"[Saved] Metrics → {metrics_path}")

---
## 17. Confusion Matrix

In [ ]:
def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: list[str],
    output_dir: str,
) -> None:
    """Display and save a normalised confusion matrix heatmap."""
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    # Shorter display names for readability
    short_names = [
        n.replace(" and ", "\n& ").replace(" And ", "\n& ")
        for n in class_names
    ]

    fig, ax = plt.subplots(figsize=(13, 11))
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=short_names,
        yticklabels=short_names,
        ax=ax,
        linewidths=0.4,
        vmin=0,
        vmax=1,
    )
    ax.set_title("Confusion Matrix (Normalised) — 12-Class Test Set", fontsize=13, pad=12)
    ax.set_ylabel("True Label", fontsize=11)
    ax.set_xlabel("Predicted Label", fontsize=11)
    plt.xticks(rotation=40, ha="right", fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()

    path = os.path.join(output_dir, "confusion_matrix.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[Saved] Confusion matrix → {path}")


plot_confusion_matrix(y_true, y_pred, class_names, cfg.OUTPUT_DIR)

---
## 18. Per-Class F1 Bar Chart

Useful for quickly spotting which genres the model struggles with most.

In [ ]:
_, _, per_class_f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, labels=list(range(num_classes))
)

f1_df = pd.DataFrame({
    "Genre":   class_names,
    "F1":      per_class_f1,
    "Support": support,
}).sort_values("F1", ascending=True)

colors = ["#d73027" if v < 0.70 else "#fee090" if v < 0.80 else "steelblue"
          for v in f1_df["F1"]]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(f1_df["Genre"], f1_df["F1"], color=colors, edgecolor="white")

for bar, (_, row) in zip(bars, f1_df.iterrows()):
    ax.text(
        bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
        f"{row['F1']:.3f}  (n={int(row['Support'])})",
        va="center", fontsize=8,
    )

ax.set_xlim(0, 1.15)
ax.axvline(0.80, color="gray", linestyle="--", alpha=0.5, label="F1=0.80")
ax.set_title("Per-Class F1-Score — Test Set", fontsize=12)
ax.set_xlabel("F1-Score")
ax.legend(fontsize=8)
plt.tight_layout()

path = os.path.join(cfg.OUTPUT_DIR, "per_class_f1.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[Saved] Per-class F1 → {path}")

---
## 19. Inference on New Texts

Use the saved best model to predict genres for arbitrary book descriptions.

In [ ]:
def predict(
    texts: list[str],
    model_dir: str = None,
    device: torch.device = None,
    return_probs: bool = False,
) -> list[str] | list[dict]:
    """
    Predict genres for a list of book descriptions using the saved best model.

    Parameters
    ----------
    texts        : list of raw book descriptions (or title + description strings)
    model_dir    : path to saved model directory (defaults to cfg.OUTPUT_DIR/best_model)
    device       : torch device (auto-detected if None)
    return_probs : if True, return top-3 genres with probabilities instead of just the top-1

    Returns
    -------
    list of predicted genre strings, or list of dicts with top-3 probabilities
    """
    if model_dir is None:
        model_dir = os.path.join(cfg.OUTPUT_DIR, "best_model")
    if device is None:
        device = get_device()

    _tokenizer = RobertaTokenizer.from_pretrained(model_dir)
    _model     = RobertaForSequenceClassification.from_pretrained(model_dir)
    _model.to(device)
    _model.eval()

    id2label    = _model.config.id2label
    results     = []

    for text in texts:
        text = clean_text(text)
        enc  = _tokenizer(
            text,
            max_length=cfg.MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        with torch.no_grad():
            logits = _model(
                input_ids=enc["input_ids"].to(device),
                attention_mask=enc["attention_mask"].to(device),
            ).logits

        probs   = torch.softmax(logits, dim=-1).squeeze()
        top3    = probs.topk(3)

        if return_probs:
            results.append({
                id2label[idx.item()]: round(score.item(), 4)
                for idx, score in zip(top3.indices, top3.values)
            })
        else:
            results.append(id2label[probs.argmax().item()])

    return results


print("✅ predict() function defined.")

In [ ]:
# ── Demo: predict genres for sample descriptions (matching the 12 new class names)
sample_texts = [
    # Speculative Fiction
    "In the year 2347, humanity must colonize a distant planet after Earth "
    "becomes uninhabitable due to climate catastrophe.",

    # Crime and Suspense
    "A detective investigating a string of murders in 1930s Chicago discovers "
    "the killer may be someone inside the police department itself.",

    # History and Biography
    "A sweeping account of the Roman Empire's decline, tracing the political "
    "decisions and military failures that led to its eventual collapse.",

    # Romance
    "Two rival chefs competing for the same Michelin star find their professional "
    "rivalry slowly turning into something neither of them expected.",

    # Mind and Self Improvement
    "A practical framework for building lasting habits through small, consistent "
    "daily changes rather than radical transformations.",

    # Cooking
    "Recipes from across the Mediterranean, exploring the flavours and techniques "
    "of Greek, Turkish, and Lebanese home cooking.",
]

# Get top-1 predictions with probabilities
predictions = predict(sample_texts, return_probs=True)

print("\n📖 Genre Predictions\n" + "═" * 65)
for text, pred in zip(sample_texts, predictions):
    top_genre, top_prob = list(pred.items())[0]
    print(f"  Description : {text[:75]}...")
    print(f"  Top genre   : {top_genre} ({top_prob*100:.1f}%)")
    others = list(pred.items())[1:]
    print(f"  Runners-up  : {others[0][0]} ({others[0][1]*100:.1f}%),  "
                         f"{others[1][0]} ({others[1][1]*100:.1f}%)")
    print()

---
## 20. Summary

In [ ]:
logger.info("\n" + "=" * 72)
logger.info("  TRAINING COMPLETE  [12-Class Dataset]")
logger.info(f"  Dataset          : {cfg.DATA_PATH}")
logger.info(f"  Text column      : {cfg.TEXT_COLUMN}")
logger.info(f"  Output directory : {os.path.abspath(cfg.OUTPUT_DIR)}")
logger.info(f"  Best model       : {os.path.abspath(best_model_path)}")
logger.info(f"  Test Accuracy    : {test_acc*100:.2f}%")
logger.info(f"  Weighted F1      : {f1:.4f}")
logger.info(f"  Macro F1         : {macro_f1:.4f}")
logger.info("=" * 72)

# List output files
print("\nOutput files:")
for fname in sorted(os.listdir(cfg.OUTPUT_DIR)):
    fpath = os.path.join(cfg.OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        print(f"  {fname:<45} {size/1024:>8.1f} KB")
    else:
        print(f"  {fname}/  (directory)")